In [ ]:
# This code extracts X% of the sampled dataset to form a validation set.
# The validation set is chosen by a random selection among the samples

In [1]:
# Cell 1 — Setup
from ase.io.trajectory import Trajectory
from ase.io import write
import numpy as np
import json, math, os, time
from pathlib import Path

In [3]:
# sampled training set

# --- User inputs ---
SRC_TRAJ = "selected/vaspdata.Ei.0.3.Ts.300.NO.rand.zpe/FPS_selected.traj"         # path to your source .traj
PERCENT  = 20                   # e.g. 10 for 10%
SEED     = 42                   # set None for non-deterministic
# output directory
OUT_DIR  = "selected/vaspdata.Ei.0.3.Ts.300.NO.rand.zpe/valset"  # output directory
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
OUT_PREFIX = f"{OUT_DIR}/FPS_selected_valset_{SEED}_{PERCENT}"           # output stem

# Outputs
OUT_TRAJ = f"{OUT_PREFIX}.traj"
OUT_XYZ  = f"{OUT_PREFIX}.xyz"
OUT_TXT  = f"{OUT_PREFIX}.txt"

# Sanity
assert 0 < PERCENT <= 100, "PERCENT must be in (0, 100]."
assert os.path.exists(SRC_TRAJ), f"Missing: {SRC_TRAJ}"

In [4]:

# Cell 2 — Inspect source trajectory and choose frames
traj = Trajectory(SRC_TRAJ, mode="r")
n_frames = len(traj)
k = max(1, int(round(PERCENT * n_frames / 100.0)))

rng = np.random.default_rng(SEED)
indices = np.sort(rng.choice(n_frames, size=k, replace=False))

print(f"Frames in source: {n_frames}")
print(f"Sampling: {k} frames ({PERCENT}%)")
print("First 10 sampled indices:", indices[:10])

Frames in source: 12263
Sampling: 2453 frames (20%)
First 10 sampled indices: [ 5 10 12 13 16 17 23 31 34 42]


In [5]:
# Cell 3 — Write selected frames to .traj and .xyz
# .traj: streamed write (no big memory spike)
if os.path.exists(OUT_TRAJ):
    os.remove(OUT_TRAJ)

with Trajectory(OUT_TRAJ, mode="w") as T:
    for i in indices:
        T.write(traj[i])

# .xyz: append mode to avoid keeping all frames in RAM
if os.path.exists(OUT_XYZ):
    os.remove(OUT_XYZ)

first = True
for i in indices:
    write(OUT_XYZ, traj[i], append=not first, format="extxyz")
    first = False

print(f"Wrote: {OUT_TRAJ}")
print(f"Wrote: {OUT_XYZ}")

Wrote: selected/vaspdata.Ei.0.3.Ts.300.NO.rand.zpe/valset/FPS_selected_valset_42_20.traj
Wrote: selected/vaspdata.Ei.0.3.Ts.300.NO.rand.zpe/valset/FPS_selected_valset_42_20.xyz


In [11]:
# Cell 4 — Manifest (.txt) with details for each selected frame
# Captures index, natoms, cell (a,b,c), pbc, energy

ts = time.strftime("%Y-%m-%d %H:%M:%S")

with open(OUT_TXT, "w", encoding="utf-8") as f:
    f.write("# Validation set manifest\n")
    f.write(f"# Source: {Path(SRC_TRAJ).resolve()}\n")
    f.write(f"# Created: {ts}\n")
    f.write(f"# Total frames: {n_frames}\n")
    f.write(f"# Percent: {PERCENT}\n")
    f.write(f"# Seed: {SEED}\n")
    f.write(f"# Selected frames: {k}\n\n")
    f.write("index\tnatoms\tcell_a\tcell_b\tcell_c\tpbc\tenergy\tforces_json\tinfo_json\n")

    for i in indices:
        atoms = traj[i]
        cell = atoms.cell.lengths()
        pbc = tuple(bool(x) for x in atoms.pbc)

        # Energy: ASE convention is atoms.info["energy"] (may vary)
        #energy = atoms.info.get("energy", "NA")
        energy = atoms.get_potential_energy()

        line = (
            f"{i}\t{len(atoms)}\t"
            f"{cell[0]:.6f}\t{cell[1]:.6f}\t{cell[2]:.6f}\t"
            f"{pbc}\t{energy}\t\n"
        )
        f.write(line)

print(f"Wrote: {OUT_TXT}")

Wrote: selected/vaspdata.Ei.0.3.Ts.300.NO.rand.zpe/valset/FPS_selected_valset_42_20.txt
